# Train the campus YOLO model (Vision Security Robot – FOCV)

Fine-tunes **YOLOv8n** (pretrained on COCO) on our auto-labelled campus dataset (classes: `person`, `suitcase`).

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

Then run the cells one by one (▶ on each cell, or Shift+Enter).

In [ ]:
# 1) Check the GPU and install Ultralytics YOLO
!nvidia-smi -L
!pip install -q ultralytics

In [ ]:
# 2) Upload dataset_for_colab.zip (from the VisionSecurityRobot folder on your laptop)
from google.colab import files
up = files.upload()
print(list(up))

In [ ]:
# 3) Unzip and write a data.yaml that points to the Colab copy
!rm -rf /content/ds && mkdir -p /content/ds && unzip -q dataset_for_colab.zip -d /content/ds
open('/content/ds/data.yaml','w').write('''path: /content/ds
train: train/images
val: val/images
test: test_night/images
names:
  0: person
  1: suitcase
''')
!ls /content/ds; !ls /content/ds/train/images | wc -l

In [ ]:
# 4) Train (about 10-20 minutes on a T4 GPU)
#    - starts from the COCO-pretrained weights (transfer learning)
#    - hsv_v: strong brightness augmentation -> robust to night images
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='/content/ds/data.yaml', epochs=60, imgsz=416, batch=32, patience=20,
            hsv_v=0.6, hsv_s=0.6, fliplr=0.5, mosaic=1.0, close_mosaic=10,
            project='/content/runs', name='campus', exist_ok=True, seed=7)

In [ ]:
# 5) Evaluate on the held-out NIGHT test set (never seen during training)
best = YOLO('/content/runs/campus/weights/best.pt')
m = best.val(data='/content/ds/data.yaml', split='test', imgsz=416)
print('mAP50 %.3f  mAP50-95 %.3f' % (m.box.map50, m.box.map))

In [ ]:
# 6) Download the trained model + training charts
!cp /content/runs/campus/weights/best.pt /content/campus_yolo.pt
!cd /content/runs && zip -qr /content/training_results.zip campus -x 'campus/weights/*'
files.download('/content/campus_yolo.pt')
files.download('/content/training_results.zip')

**After downloading:** move `campus_yolo.pt` into the `models` folder of VisionSecurityRobot, and unzip `training_results.zip` into `VisionSecurityRobot/training/results/`.